In [1]:
#Upload the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

In [ ]:
# Choose the  CSV file from your computer
from google.colab import files

uploaded = files.upload()
csv_file = next(iter(uploaded))
df = pd.read_csv(csv_file)

print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
#Univariate analysis
print(df["Outcome"].value_counts(normalize =True)*100)

In [ ]:
sns.countplot(data = df, x = "Outcome")
plt.title("Distribution of Diabetes")
plt.show()

In [ ]:
sns.countplot(data = df, x = "Pregnancies")
plt.title("Distribution of Pregnancies")
plt.show()

In [ ]:
#Bivariate
sns.boxplot(data = df, x = "Outcome", y="Age")
plt.title("Distribution of Diabetes")
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include = "number")

In [ ]:
# Multivariate Analysis
plt.figure(figsize = (10, 8))
sns.heatmap(numeric_df.corr(), annot = True, cmap = "coolwarm",fmt= ".2f")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include = ["int64","float64"]).columns

In [ ]:
# Outlier detection
sns.boxplot(data =df[numeric_df])
plt.xticks(rotation = 90)
plt.show()

In [ ]:
# Treat  the Outliers
for col in numeric_df:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  count = ((df[col] < lower_bound) |
          (df[col] > upper_bound)).sum()
  print(col,":", count)

In [ ]:
# Treating outliers
outlier_cols = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "Pedigree",
    "Age"
]

In [ ]:
# Apply the IQR
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3-Q1
    lower_bound = Q1 - 1.5*IQR
    upper_bound = Q3 + 1.5*IQR

    # cap values
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

In [ ]:
fig, axes = plt.subplots(len(outlier_cols),1, figsize = (8,20))
for i, col in enumerate(outlier_cols):
    sns.boxplot(data = df, x = col, ax = axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop("Outcome", axis = 1)
y = df["Outcome"]

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
print(X_train.shape, X_test.shape,y_train.shape, y_test.shape)

In [ ]:
from imblearn.over_sampling import SMOTE
print("Before Smote:\n",y_train.value_counts())
smote = SMOTE(random_state = 42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("After Smote:\n",y_train_resampled.value_counts())

In [ ]:
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_resampled, y_train_resampled)


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
sns.heatmap(conf_matrix, annot = True, fmt = "d", cmap = "Blues")
plt.title("Confusion Matrix")
plt.show()
print("accuracy:", accuracy)
print("precision:", precision)
print("recall:", recall)
print("f1 score:", f1)
print("classification report:\n", class_report)

## Observation:

In this project, I used a Diabetes Dataset containing 768 records and 8 input features. The target column is Outcome, where 0 represents a non-diabetic person and 1 represents a diabetic person. I observed that the dataset has more non-diabetic records than diabetic records.

During data analysis, I found that Glucose level, BMI, Age, and other health factors are useful for diabetes prediction. I also found some outliers in features like BloodPressure, Insulin, BMI, Pedigree, and Age. These outliers were checked using the IQR method and treated before training the model.

The dataset was divided into 80% training data and 20% testing data. Since the number of diabetic and non-diabetic cases was not equal, I used SMOTE to balance the training data. After that, I used StandardScaler to bring all the feature values to a similar scale.

For prediction, I used the Logistic Regression algorithm. The model achieved an accuracy of 68.83%. For diabetic cases, the model achieved 54.79% precision and 72.73% recall. This means the model was able to detect most of the people who actually had diabetes, but it also made some incorrect predictions.

The F1-score was 62.50%, which shows a reasonable balance between precision and recall. Overall, the model gave useful results for basic diabetes prediction, but its performance can be improved by using more data, better features, and trying other machine learning algorithms.

Through this project, I learned how to perform data cleaning, data analysis, outlier treatment, data balancing, feature scaling, model training, and model evaluation.
